WARNING: Notebook was written during development. May not work as intended. Retained here only for completion and future reference.

Twist a clamped-clamped rod through many full turns.

Nodal rotations are stored as Rodrigues vectors measured from a fixed reference. The
exponential map degenerates whenever a rotation reaches an integer multiple of 2 pi,
where its right Jacobian collapses to rank one and the rod can no longer equilibrate
a transverse moment. Wrapping each converged rotation back into ‖θ‖ ≤ π moves every
degree of freedom off that singular set without changing the configuration.

Set `num_turns` and rerun. Start at 0.5 to confirm nothing regressed, then 1.5 for
the first wrap, then 5, then 27.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import crops.src.bc as _bc
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz

jax.config.update("jax_enable_x64", True)
plt.rcParams.update(_viz.default_plot_settings)

### Rod

In [ ]:
num_turns = 10
rod_length = 100.0
section_radius = 0.35
shear_modulus = 3.3e5

# The log map used to extract element curvature always returns ‖θ‖ ≤ π, so a relative
# rotation of more than half a turn across a single element comes back as its
# complement and the recovered twist silently flips sign. Resolving `num_turns` needs
# more than twice that many elements.
num_elems = max(400, int(np.ceil(8.0 * num_turns)))
num_nodes = num_elems + 1

# Four fibers matched to a circular section of the same area and second moment.
area_target = np.pi * section_radius**2
moi_target = np.pi * section_radius**4 / 4.0
area_per_fiber = area_target / 4.0
fiber_offset = np.sqrt(moi_target / (2.0 * area_per_fiber))
offsets_local = np.array(
  [[fiber_offset, 0.0], [-fiber_offset, 0.0], [0.0, fiber_offset], [0.0, -fiber_offset]]
)
num_fibers = 4

rod = _mesher.build_rod_geometry(
  analytical_curve=lambda s: jnp.array([s, 0.0, 0.0]),
  parametric_coords=jnp.linspace(0.0, rod_length, num_nodes),
  fiber_offsets=jnp.asarray(np.tile(offsets_local, (num_elems, 1, 1))),
  fiber_areas=jnp.full((num_elems, num_fibers), area_per_fiber),
  fiber_mass_density=jnp.ones((num_elems, num_fibers)),
  fiber_centroid_moi=jnp.zeros((num_elems, num_fibers, 2)),
  seed_binormal=jnp.array([0.0, 0.0, 1.0]),
  initial_bow_ratio=1.0e-3,
)
total_twist = 2.0 * np.pi * num_turns
print(f"{num_turns} turns = {total_twist:.4f} rad prescribed at the far end")
per_elem = np.degrees(total_twist / num_elems)
print(f"{num_elems} elements, {per_elem:.3f} deg per element")

### Boundary conditions

In [ ]:
num_steps = max(100, int(np.ceil(20.0 * num_turns)))
twist_ramp = np.linspace(total_twist / num_steps, total_twist, num_steps)

root, tip = np.array([0]), np.array([num_nodes - 1])
clamped_dofs = (
  _utils.Translation.X,
  _utils.Translation.Y,
  _utils.Translation.Z,
  _utils.Rotation.THETA_X,
  _utils.Rotation.THETA_Y,
  _utils.Rotation.THETA_Z,
)
boundary_list = [
  _bc.DirichletBC(
    values=(dof, [np.array([0.0]) for _ in range(num_steps)]),
    node_targets=[root for _ in range(num_steps)],
    name=f"root_{dof}",
  )
  for dof in clamped_dofs
]

# The far end is held at its reference position and rotated about the rod axis. With
# both ends pinned at the natural length the rod is in tension as it tries to helix,
# so it stays straight and the twist distributes uniformly. That makes the recovered
# curvature exactly checkable before any buckling is introduced.
boundary_list += [
  _bc.DirichletBC(
    values=(dof, [np.array([0.0]) for _ in range(num_steps)]),
    node_targets=[tip for _ in range(num_steps)],
    name=f"tip_{dof}",
  )
  for dof in (
    _utils.Translation.Y,
    _utils.Translation.Z,
    _utils.Rotation.THETA_Y,
    _utils.Rotation.THETA_Z,
  )
]
boundary_list += [
  _bc.DirichletBC(
    values=(_utils.Translation.X, [np.array([0.0]) for _ in range(num_steps)]),
    node_targets=[tip for _ in range(num_steps)],
    name="tip_hold_x",
  ),
  _bc.DirichletBC(
    values=(_utils.Rotation.THETA_X, [np.array([a]) for a in twist_ramp]),
    node_targets=[tip for _ in range(num_steps)],
    name="tip_twist",
  ),
]
bc = _bc.process_boundary_conditions(rod, boundary_list)

### Solve

In [ ]:
nr_solver = _solver.NewtonRaphson(
  solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
  nonlinear_settings={
    "max_iter": 200,
    "threshold": 1e-6,
    "abs_threshold": 1e-6,
    "lam_min": 0.01,
    "line_search_max_iter": 15,
    "line_search_shrink": 0.5,
    "line_search_alpha_min": 1e-6,
    "line_search_armijo_c": 1e-4,
  },
  linear_settings={"solver": _solver.LinearSolvers.SCIPY_SPARSE, "rtol": 1e-10},
)
problem = _fe.CosseratProblem(rod=rod, bc=bc, nr_solver=nr_solver)

u_star, u_history = problem.solve(
  u0=jnp.zeros(rod.num_dofs),
  shear_moduli=jnp.full((num_elems, num_fibers), shear_modulus),
)
history = np.asarray(u_history).reshape(num_steps, num_nodes, -1)

### Verification

In [ ]:
coords, directors, _ = _mesher.extract_coords_and_director_from_deformation(rod, u_star)
curvature = np.asarray(
  _mesher.compute_discrete_cosserat_kinematics(
    node_coords=coords, directors=directors, arc_coords=rod.arc_coords
  ).curvature
)

# Twist per unit reference arc length is the prescribed rotation spread over the rod.
twist_exact = total_twist / rod_length
twist_achieved = curvature[:, _mesher.CurvatureComponent.TWIST]

# Turns cannot be read off the stored rotation vectors once they are wrapped, but the
# curvature integral is a frame-invariant line integral and is immune to that.
turns_recovered = float(
  np.sum(twist_achieved * np.asarray(rod.elem_arc_lengths)) / (2.0 * np.pi)
)
nodal = np.asarray(u_star).reshape(num_nodes, -1)
stored_rotation = np.degrees(np.abs(nodal[:, 3:]).max())

print(f"kappa_3 exact      = {twist_exact:.6e} 1/m")
print(f"kappa_3 mean       = {twist_achieved.mean():.6e} 1/m")
print(
  f"relative error     = "
  f"{abs(twist_achieved.mean() - twist_exact) / twist_exact:.3%}"
)
print(f"turns from integral= {turns_recovered:.6f}   (prescribed {num_turns})")
print(f"max stored ‖theta‖ = {stored_rotation:.1f} deg   (wrapped, so <= 180)")

In [ ]:
fig, (ax_twist, ax_rot) = plt.subplots(1, 2, figsize=(11, 4))
ax_twist.plot(np.asarray(rod.elem_centers)[:, 0], twist_achieved, lw=1.5)
ax_twist.axhline(twist_exact, color="k", ls="--", lw=1.2, label=r"$\Phi / L$")
ax_twist.set_xlabel(r"$\hat{s}$ [m]")
ax_twist.set_ylabel(r"$\kappa_3$ [1/m]")

ax_twist.legend()

# The stored rotation is what the wrap acts on. Without it this would grow without
# bound and cross 2 pi; with it, it sawtooths inside [-180, 180] deg.
ax_rot.plot(np.degrees(history[:, -1, 3]), "o-", ms=3, label="tip, stored")
ax_rot.plot(np.degrees(twist_ramp), "k--", lw=1, label="tip, prescribed")
ax_rot.axhline(180.0, color="r", ls=":", lw=1)
ax_rot.axhline(-180.0, color="r", ls=":", lw=1)
ax_rot.set_xlabel("load step")
ax_rot.set_ylabel("rotation [deg]")

ax_rot.legend(fontsize=8)
fig.tight_layout()
plt.show()

### Helical buckling

Releasing the far end axially by a slack lets the rod shorten, which puts it into
compression and allows the twist to trade for bending. Past a critical twist it
buckles into a localized helix.

Note that the fiber-bundle neo-Hookean forces E = 3G, so the torsional and bending
stiffnesses cannot be set independently. Gazzola's localized helical buckling case
uses G/E = 0.01, which this constitutive model cannot represent, so the buckling
threshold will not match theirs numerically even though the mechanism is the same.

In [ ]:
slack = 0.05

buckling_bc = [b for b in boundary_list if b.name != "tip_hold_x"]
buckling_bc += [
  _bc.DirichletBC(
    values=(
      _utils.Translation.X,
      [np.array([-slack * (k + 1) / num_steps]) for k in range(num_steps)],
    ),
    node_targets=[tip for _ in range(num_steps)],
    name="tip_slack",
  )
]
buckling_problem = _fe.CosseratProblem(
  rod=rod,
  bc=_bc.process_boundary_conditions(rod, buckling_bc),
  nr_solver=nr_solver,
)
u_buckled, buckled_history = buckling_problem.solve(
  u0=jnp.zeros(rod.num_dofs),
  shear_moduli=jnp.full((num_elems, num_fibers), shear_modulus),
)
buckled_nodal = np.asarray(u_buckled).reshape(num_nodes, -1)
buckled = np.asarray(rod.node_coords) + buckled_nodal[:, :3]

fig = plt.figure(figsize=(10, 4))
ax = fig.add_subplot(projection="3d")
ax.plot(buckled[:, 0], buckled[:, 1], buckled[:, 2], lw=1.5)
ax.set_xlabel("$x$ [m]")
ax.set_ylabel("$y$ [m]")
ax.set_zlabel("$z$ [m]")
ax.set_ylim(-0.1 * rod_length, 0.1 * rod_length)
ax.set_zlim(-0.1 * rod_length, 0.1 * rod_length)
plt.show()

print(
  f"max stored ‖theta‖ = " f"{np.degrees(np.abs(buckled_nodal[:, 3:]).max()):.1f} deg"
)
print(f"peak lateral excursion = " f"{np.abs(buckled[:, 1:]).max():.4f} m")